# Geo Experiment — UK Postcode Split

This notebook creates a Treatment / Control split for a UK geo experiment.

**UK postcode hierarchy we use:**

| Level | Example | What it means |
|-------|---------|---------------|
| Full postcode | `NW3 4ED` | Individual street |
| **District** | `NW3` | Our primary unit — outward code before the space |
| **Area** | `NW` | Grouping of districts — letters only |

**Steps:**
1. Load and clean Braintree data (UK transactions only)
2. Extract postcode district (`NW3`) and area (`NW`)
3. Aggregate to weekly bookings per district
4. Cluster districts by booking behaviour
5. Split clusters into Treatment and Control
6. Validate the split with trend charts

## Step 0 — Setup
Run this cell first every time you open the notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Repo root is one level above the notebooks/ folder
BASE_DIR  = os.path.dirname(os.path.abspath('.'))
DATA_DIR  = os.path.join(BASE_DIR, 'data')
PLOTS_DIR = os.path.join(BASE_DIR, 'plots')
os.makedirs(DATA_DIR,  exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
print('Ready.')


---
## ⚙️ Config — change settings here

This is the only cell you need to edit to adjust the split.
All steps below will automatically use these values.

In [ ]:
# ── Number of clusters ────────────────────────────────────────────────────
# Higher = more granular grouping. Start with 4, adjust after seeing Step 5 scatter.
N_CLUSTERS = 4

# ── Pre-period length for normalisation ───────────────────────────────────
# Each area is normalised to its own mean over the first N weeks of data.
# This removes each area's unique baseline level before comparing trends.
PRE_PERIOD_WEEKS = 8

# ── Areas to exclude from the split ───────────────────────────────────────
# Add area codes here (e.g. 'BT' for Northern Ireland, 'HS' for Outer Hebrides).
# Leave empty [] to keep all areas.
EXCLUDE_AREAS = []

print(f'N_CLUSTERS       : {N_CLUSTERS}')
print(f'PRE_PERIOD_WEEKS : {PRE_PERIOD_WEEKS}')
print(f'Exclude areas    : {EXCLUDE_AREAS if EXCLUDE_AREAS else "none"}')

---
## Step 1 — Load Braintree data

Same export as the US notebook — we filter to UK transactions next.

In [ ]:
BRAINTREE_CSV = os.path.join(BASE_DIR, 'Braintree data - March 2026 - Sheet1.csv')

raw = pd.read_csv(BRAINTREE_CSV, dtype=str)
raw.columns = raw.columns.str.strip()

print(f'Total rows loaded : {len(raw):,}')
print(f'Countries in data : {raw["Billing Country"].str.strip().value_counts().head(10).to_string()}')


---
## Step 2 — Filter to UK and clean

We keep **United Kingdom** transactions only and remove refunds/credits.

We also extract the **postcode district** (`NW3`) and **area** (`NW`) from the full postcode.
UK postcodes always follow the pattern `OUTWARD INWARD` — we take everything before the space.

In [ ]:
# Keep UK only
uk = raw[raw['Billing Country'].str.strip() == 'United Kingdom'].copy()

# Remove refunds / credits
uk = uk[uk['Transaction Type'].str.strip().str.lower() != 'credit'].copy()

# Parse date → Monday of each week
uk['date'] = pd.to_datetime(uk['Created Datetime'], format='%m/%d/%Y %H:%M:%S', errors='coerce')
uk['week'] = (uk['date'] - pd.to_timedelta(uk['date'].dt.dayofweek, unit='d')).dt.normalize()

# Parse dollar amount
uk['Amount Authorized'] = pd.to_numeric(uk['Amount Authorized'], errors='coerce')

# Clean postcode: uppercase, strip whitespace
uk['postcode_raw'] = uk['Billing Postal Code'].str.strip().str.upper()

# Extract district = outward code (everything before the last space or last 3 chars)
# UK inward code is always exactly 3 characters (digit + 2 letters)
uk['district'] = uk['postcode_raw'].str.replace(r'\s+', ' ', regex=True).str.extract(r'^([A-Z]{1,2}[0-9][0-9A-Z]?)')

# Extract area = leading letters only
uk['area'] = uk['district'].str.extract(r'^([A-Z]{1,2})')

# Drop rows with unparseable postcodes
uk = uk.dropna(subset=['district', 'area'])

print(f'UK transactions kept   : {len(uk):,}')
print(f'Date range             : {uk["date"].min().date()}  to  {uk["date"].max().date()}')
print(f'Unique districts       : {uk["district"].nunique():,}')
print(f'Unique areas           : {uk["area"].nunique():,}')

# Preview postcode parsing
uk[['postcode_raw', 'district', 'area']].drop_duplicates().head(10)

---
## Step 3 — Roll up to weekly bookings per area

We aggregate all transactions up to the **area level** (`NW`, `SW`, `E`, etc.) — one row per week per area.

In [ ]:
# Weekly totals at area level
weekly = (
    uk.groupby(['week', 'area'])
    .agg(
        bookings = ('Transaction ID', 'nunique'),
        sales    = ('Amount Authorized', 'sum'),
    )
    .reset_index()
)

print(f'Unique weeks : {weekly["week"].nunique()}')
print(f'Unique areas : {weekly["area"].nunique()}')
weekly.head(8)

---
## Step 4 — Overview by area

Let's see how bookings are distributed across areas before we split them.

In [ ]:
area_totals = (
    weekly.groupby('area')['bookings']
    .sum()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(9, max(4, len(area_totals) * 0.28)))
ax.barh(area_totals.index, area_totals.values, color='steelblue')
ax.set_xlabel('Total bookings')
ax.set_title('Total bookings by area')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

print(f'Total areas : {len(area_totals)}')
print(area_totals.sort_values(ascending=False).to_string())

---
## Step 5 — Cluster areas by booking behaviour

We group areas into clusters based on:
- **Average weekly bookings** — the overall size of the area
- **Variability** — how much bookings fluctuate week to week

Areas in the same cluster behave similarly. We split within clusters to keep the T/C groups balanced.

In [ ]:
# One row per area with summary stats
# Areas in EXCLUDE_AREAS are removed before clustering
area_stats = (
    weekly[~weekly['area'].isin(EXCLUDE_AREAS)]
    .groupby('area')['bookings']
    .agg(mean_bookings='mean', std_bookings='std', total_bookings='sum')
    .reset_index()
    .fillna(0)
)

if EXCLUDE_AREAS:
    print(f'Excluded areas : {EXCLUDE_AREAS}')
    print(f'Areas remaining: {len(area_stats)}')

# Cluster on standardised mean and std — uses N_CLUSTERS from config
X = StandardScaler().fit_transform(area_stats[['mean_bookings', 'std_bookings']])
area_stats['cluster'] = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10).fit_predict(X)
area_stats['cluster'] = area_stats['cluster'].astype(str)

print('Areas per cluster:')
print(area_stats.groupby('cluster').agg(
    n_areas=('area', 'count'),
    avg_weekly_bookings=('mean_bookings', 'mean'),
    areas=('area', lambda x: ', '.join(sorted(x)))
).round(1).to_string())

# Scatter: mean vs std coloured by cluster
colours = plt.cm.Set1.colors
fig, ax = plt.subplots(figsize=(9, 5))
for i, cid in enumerate(sorted(area_stats['cluster'].unique())):
    sub = area_stats[area_stats['cluster'] == cid]
    ax.scatter(sub['mean_bookings'], sub['std_bookings'],
               label=f'Cluster {cid}  (n={len(sub)})',
               color=colours[i % len(colours)], alpha=0.8, s=60)
    for _, row in sub.iterrows():
        ax.annotate(row['area'], (row['mean_bookings'], row['std_bookings']),
                    fontsize=7, alpha=0.7, xytext=(4, 2), textcoords='offset points')

ax.set_xlabel('Mean weekly bookings per area')
ax.set_ylabel('Std deviation of weekly bookings')
ax.set_title(f'Area clusters (N={N_CLUSTERS}) — size vs variability')
ax.legend()
plt.tight_layout()
plt.show()

---
## Step 6 — Assign Treatment and Control

Within each cluster, we randomly assign half the areas to Treatment and half to Control.

In [ ]:
np.random.seed(42)
assignment = {}
for _, group in area_stats.groupby('cluster'):
    areas = group['area'].values
    treat = set(np.random.choice(areas, len(areas) // 2, replace=False))
    for a in areas:
        assignment[a] = 'Treatment' if a in treat else 'Control'

area_stats['assignment'] = area_stats['area'].map(assignment)
weekly['assignment']     = weekly['area'].map(assignment)

counts = area_stats['assignment'].value_counts()
print(f'Treatment areas : {counts["Treatment"]}')
print(f'Control areas   : {counts["Control"]}')

# Balance summary
balance = (
    weekly.groupby('assignment')
    .agg(n_areas=('area', 'nunique'),
         total_bookings=('bookings', 'sum'),
         avg_per_week=('bookings', 'mean'))
    .reset_index()
)
print('\nBalance check:')
print(balance.to_string(index=False))

# Show which area landed where
print('\nFull area assignment:')
print(area_stats[['area', 'cluster', 'assignment', 'mean_bookings', 'total_bookings']]
      .sort_values(['assignment', 'cluster'])
      .round(1).to_string(index=False))

---
## Step 7 — Normalise each area to its pre-period baseline

Before comparing trends across areas, we normalise each area's bookings by its own
**pre-period mean** (first `PRE_PERIOD_WEEKS` weeks).

**Why this matters:** A large area like `SW` has 10× more bookings than a small area like `AB`.
Without normalisation, large areas dominate the group trend and hide what's happening in smaller ones.
After normalisation, every area contributes equally — the trend line shows the *average relative movement*
across all areas, accounting for each area's own growth trajectory and seasonal pattern.

In [ ]:
pre_weeks = sorted(weekly['week'].unique())[:PRE_PERIOD_WEEKS]

# Compute each area's mean bookings over the pre-period
pre_means = (
    weekly[weekly['week'].isin(pre_weeks)]
    .groupby('area')['bookings']
    .mean()
    .rename('pre_mean')
    .reset_index()
)

# Attach pre-period mean to every row, then normalise
# bookings_norm = 100 means 'performing at pre-period average'
# bookings_norm = 110 means '10% above the area's own pre-period average'
weekly = weekly.merge(pre_means, on='area', how='left')
weekly['bookings_norm'] = weekly['bookings'] / weekly['pre_mean'] * 100

print(f'Pre-period weeks used : {[str(w.date()) for w in pre_weeks]}')
print(f'\nSample — first 3 areas normalised:')
print(
    weekly[weekly['week'].isin(pre_weeks[:3])]
    .groupby('area')[['bookings', 'pre_mean', 'bookings_norm']]
    .mean().round(2).head(6).to_string()
)

---
## Step 8 — Weekly trend: Treatment vs Control (normalised)

Each area is first normalised to its own 8-week pre-period mean.
The line shows the **average normalised index across all areas** in each group.
A value of 100 = performing at pre-period average. Lines tracking together = balanced split.

In [ ]:
# Average normalised index per group per week
weekly_group = (
    weekly.groupby(['week', 'assignment'])['bookings_norm']
    .mean()   # mean across areas — each area contributes equally
    .reset_index()
    .rename(columns={'bookings_norm': 'index'})
)

fig, ax = plt.subplots(figsize=(12, 4))

for grp, col in [('Treatment', 'tomato'), ('Control', 'steelblue')]:
    sub = weekly_group[weekly_group['assignment'] == grp]
    ax.plot(sub['week'], sub['index'], color=col, label=grp, linewidth=1.8)

# Shade the pre-period
ax.axvspan(weekly_group['week'].min(), pre_weeks[-1],
           alpha=0.08, color='grey', label=f'Pre-period ({PRE_PERIOD_WEEKS}w baseline)')
ax.axhline(100, color='grey', linestyle=':', linewidth=0.8)

ax.set_ylabel('Normalised index  (100 = pre-period average)')
ax.set_title('Weekly trend — Treatment vs Control\n'
             'Each area normalised to its own pre-period mean before aggregating')
ax.legend()
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

---
## Step 9 — Trend by cluster (normalised)

One panel per cluster. Each area is still normalised to its own pre-period mean,
so within each cluster panel we see the average relative movement — not absolute volumes.

In [ ]:
clusters = sorted(area_stats['cluster'].unique())  # auto-reflects N_CLUSTERS
n_cols = min(len(clusters), 4)
n_rows = (len(clusters) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows), squeeze=False)
axes_flat = axes.flatten()

for idx, cid in enumerate(clusters):
    ax = axes_flat[idx]
    cluster_areas = area_stats[area_stats['cluster'] == cid]['area']
    cluster_data  = weekly[weekly['area'].isin(cluster_areas)]

    grp_weekly = (
        cluster_data.groupby(['week', 'assignment'])['bookings_norm']
        .mean().reset_index()
    )

    area_labels = ', '.join(sorted(cluster_areas))
    for grp, col in [('Treatment', 'tomato'), ('Control', 'steelblue')]:
        sub = grp_weekly[grp_weekly['assignment'] == grp]
        if sub.empty:
            continue
        ax.plot(sub['week'], sub['bookings_norm'], color=col, label=grp, linewidth=1.5)

    ax.axvspan(weekly['week'].min(), pre_weeks[-1], alpha=0.08, color='grey')
    ax.axhline(100, color='grey', linestyle=':', linewidth=0.8)
    ax.set_title(f'Cluster {cid}\n{area_labels}', fontsize=9)
    ax.set_ylabel('Normalised index')
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=30)

for idx in range(len(clusters), len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle(f'Normalised trend by cluster (N={N_CLUSTERS}) — Treatment vs Control', fontsize=12)
plt.tight_layout()
plt.show()

---
## Step 9 — Save the final split

Export the area assignment to CSV for campaign targeting.

In [ ]:
final_split = (
    weekly.groupby(['area', 'assignment'])
    .agg(total_bookings=('bookings', 'sum'), avg_weekly=('bookings', 'mean'))
    .reset_index()
    .sort_values(['assignment', 'total_bookings'], ascending=[True, False])
    .round(1)
)

out_path = os.path.join(DATA_DIR, 'uk_area_split.csv')
final_split.to_csv(out_path, index=False)
print(f'Saved to: {out_path}')
print(f'\nTreatment areas : {(final_split["assignment"] == "Treatment").sum()}')
print(f'Control areas   : {(final_split["assignment"] == "Control").sum()}')
final_split

---
## Step 10 — Placebo simulation

We slide an 8-week window across the full history using the **fixed T/C assignment**.
Because no treatment was applied, any difference we measure is purely natural variance.
The resulting distribution tells us what fluctuations to expect by chance alone.

In [ ]:
WINDOW = 8   # weeks
N_SIMS  = 500

# ── All available week indices ───────────────────────────────────────────────
all_weeks = sorted(weekly['week'].unique())
n_weeks   = len(all_weeks)

placebo_diffs = []

for start in range(n_weeks - WINDOW + 1):
    window_weeks = all_weeks[start : start + WINDOW]
    w = weekly[weekly['week'].isin(window_weeks)]

    # Per-window pre-period mean for each area (first week of window = baseline)
    base = w[w['week'] == window_weeks[0]].set_index('area')['bookings_norm']

    # Normalise each area to its own first-week value in this window
    def norm_window(row):
        b = base.get(row['area'], np.nan)
        return row['bookings_norm'] / b * 100 if b and b > 0 else np.nan

    w = w.copy()
    w['win_norm'] = w.apply(norm_window, axis=1)

    # Average over window per group
    grp = w.groupby('assignment')['win_norm'].mean()
    if 'Treatment' in grp.index and 'Control' in grp.index:
        placebo_diffs.append(grp['Treatment'] - grp['Control'])

placebo_diffs = np.array(placebo_diffs)

p5,  p95  = np.percentile(placebo_diffs, [5,  95])
p25, p75  = np.percentile(placebo_diffs, [25, 75])
med       = np.median(placebo_diffs)

print(f'Windows evaluated : {len(placebo_diffs)}')
print(f'Median diff       : {med:.2f} index pts')
print(f'90 % CI           : [{p5:.2f}, {p95:.2f}]')
print(f'IQR               : [{p25:.2f}, {p75:.2f}]')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(placebo_diffs, bins=20, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(0,   color='black',  linestyle='--', linewidth=1.2, label='zero')
ax.axvline(p5,  color='tomato', linestyle=':',  linewidth=1.2, label=f'5th pct ({p5:.1f})')
ax.axvline(p95, color='tomato', linestyle=':',  linewidth=1.2, label=f'95th pct ({p95:.1f})')
ax.axvline(med, color='gold',   linestyle='-',  linewidth=1.5, label=f'median ({med:.1f})')
ax.set_xlabel('Treatment index − Control index (index pts)', fontsize=11)
ax.set_ylabel('Number of 8-week windows', fontsize=11)
ax.set_title('Placebo simulation — natural variance between T and C', fontsize=13)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'uk_placebo_sim.png'), dpi=150)
plt.show()
print('Saved: uk_placebo_sim.png')

---
## Step 11 — Power simulation

For each combination of **effect size** and **test duration** we apply a synthetic
multiplicative lift to Treatment areas, run a Welch t-test on per-area average
normalised bookings, and count how often we correctly reject H₀ (p < 0.05).

A cell is **green** when power ≥ 80 % — the conventional threshold for a well-powered study.

In [ ]:
from scipy import stats
import matplotlib.colors as mcolors

EFFECT_SIZES = [0.05, 0.10, 0.15, 0.20, 0.30]   # +5 % … +30 %
TEST_WEEKS   = [4, 6, 8, 12]
N_POWER_SIMS = 500
ALPHA        = 0.05

power_matrix = np.zeros((len(EFFECT_SIZES), len(TEST_WEEKS)))

for ei, eff in enumerate(EFFECT_SIZES):
    for wi, tw in enumerate(TEST_WEEKS):
        reject = 0
        valid_starts = list(range(n_weeks - tw))
        starts = np.random.choice(valid_starts, size=N_POWER_SIMS, replace=True)

        for start in starts:
            sim_weeks = all_weeks[start : start + tw]
            w = weekly[weekly['week'].isin(sim_weeks)].copy()

            # ── Within-window normalisation (same as placebo) ────────────────
            # Each area is indexed to its own value in the first week of the
            # window. This removes any pre-existing level drift so the t-test
            # only picks up the synthetic lift we add, not background divergence.
            first_week = sim_weeks[0]
            base = (w[w['week'] == first_week]
                    .set_index('area')['bookings_norm'])
            w['win_norm'] = w.apply(
                lambda r: r['bookings_norm'] / base.get(r['area'], np.nan) * 100,
                axis=1
            )
            w = w.dropna(subset=['win_norm'])

            # ── Apply synthetic lift to Treatment areas ───────────────────────
            w.loc[w['assignment'] == 'Treatment', 'win_norm'] *= (1 + eff)

            # ── Per-area mean over the test window ───────────────────────────
            area_means = w.groupby(['area', 'assignment'])['win_norm'].mean().reset_index()
            t_vals = area_means[area_means['assignment'] == 'Treatment']['win_norm'].values
            c_vals = area_means[area_means['assignment'] == 'Control'  ]['win_norm'].values

            if len(t_vals) >= 2 and len(c_vals) >= 2:
                _, p = stats.ttest_ind(t_vals, c_vals, equal_var=False)
                if p < ALPHA:
                    reject += 1

        power_matrix[ei, wi] = reject / N_POWER_SIMS

# ── Plot ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

cmap = mcolors.LinearSegmentedColormap.from_list('rg', ['#d73027','#fee08b','#1a9850'])
im   = ax.imshow(power_matrix, vmin=0, vmax=1, cmap=cmap, aspect='auto')

ax.set_xticks(range(len(TEST_WEEKS)))
ax.set_xticklabels([f'{w}w' for w in TEST_WEEKS], fontsize=11)
ax.set_yticks(range(len(EFFECT_SIZES)))
ax.set_yticklabels([f'+{int(e*100)}%' for e in EFFECT_SIZES], fontsize=11)
ax.set_xlabel('Test duration', fontsize=12)
ax.set_ylabel('Effect size (lift)', fontsize=12)
ax.set_title('Power matrix — probability of detecting a true lift (p < 0.05)', fontsize=13)

for ei in range(len(EFFECT_SIZES)):
    for wi in range(len(TEST_WEEKS)):
        pwr = power_matrix[ei, wi]
        colour = 'white' if pwr < 0.4 or pwr > 0.85 else 'black'
        ax.text(wi, ei, f'{pwr:.0%}', ha='center', va='center',
                fontsize=12, fontweight='bold', color=colour)

plt.colorbar(im, ax=ax, label='Power')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'uk_power_matrix.png'), dpi=150)
plt.show()
print('Saved: uk_power_matrix.png')
print('\nNote: green cells (≥ 80%) indicate a well-powered experiment.')
